# 📖 Notebook 2: Throughput and Capacity Planning

Latency tells you how long **one operation** takes.
Throughput tells you how many operations you can do **per second**.
Together, they let you answer the most important system design question:

> **"How many servers do I actually need?"**

## Learning Objectives

By the end of this notebook, you'll be able to:
- Convert daily active users (DAU) into queries per second (QPS)
- Calculate storage requirements for any system
- Estimate bandwidth needs
- Contrast three approaches: BAD (guessing) → BETTER (envelope math) → BEST (benchmarking)

## 🛠️ Setup

```bash
cd core-concepts/numbers-to-know
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import time
import json
from concurrent.futures import ThreadPoolExecutor

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "numbers_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db_connection(); conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL: {e}\n   Run: docker-compose up -d")

try:
    r = get_redis_client(); r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis: {e}\n   Run: docker-compose up -d")

## 📐 Essential Units and Conversions

Before we calculate anything, let's nail down the units.

### Powers of 2 (Memory / Storage)
```
1 KB  = 1,024 bytes       ≈ 1 thousand bytes     (a short email)
1 MB  = 1,024 KB          ≈ 1 million bytes       (a photo)
1 GB  = 1,024 MB          ≈ 1 billion bytes       (a movie)
1 TB  = 1,024 GB          ≈ 1 trillion bytes      (a small database)
1 PB  = 1,024 TB          ≈ 1 quadrillion bytes   (YouTube's daily uploads)
```

### Time Conversions (for QPS)
```
1 day    = 86,400 seconds  ≈ 100K seconds   (use 100K for easy math)
1 month  = 2.6M seconds    ≈ 2.5M seconds
1 year   = 31.5M seconds   ≈ 30M seconds
```

### Quick QPS Rules
```
1 million requests/day    ≈ 12 QPS
10 million requests/day   ≈ 120 QPS
100 million requests/day  ≈ 1,200 QPS
1 billion requests/day    ≈ 12,000 QPS
```

In [ ]:
# Let's build a quick conversion helper

def requests_to_qps(requests_per_day):
    """Convert daily requests to queries per second."""
    return requests_per_day / 86_400

def qps_to_peak(avg_qps, peak_multiplier=3):
    """Estimate peak QPS (typically 2-3× average)."""
    return avg_qps * peak_multiplier

def storage_per_day(items_per_day, bytes_per_item):
    """Calculate daily storage in human-readable format."""
    total_bytes = items_per_day * bytes_per_item
    if total_bytes >= 1e15:
        return f"{total_bytes / 1e15:.1f} PB"
    elif total_bytes >= 1e12:
        return f"{total_bytes / 1e12:.1f} TB"
    elif total_bytes >= 1e9:
        return f"{total_bytes / 1e9:.1f} GB"
    elif total_bytes >= 1e6:
        return f"{total_bytes / 1e6:.1f} MB"
    else:
        return f"{total_bytes / 1e3:.1f} KB"

def bandwidth(qps, response_size_bytes):
    """Calculate bandwidth in Mbps."""
    bits_per_sec = qps * response_size_bytes * 8
    return bits_per_sec / 1e6  # Mbps

# Demonstrate
print("📐 Quick QPS Conversions")
print("=" * 50)
for daily in [1_000_000, 10_000_000, 100_000_000, 1_000_000_000]:
    avg = requests_to_qps(daily)
    peak = qps_to_peak(avg)
    print(f"  {daily:>15,} req/day → {avg:>8,.0f} QPS avg → {peak:>8,.0f} QPS peak")

print()
print("📐 Quick Storage Conversions")
print("=" * 50)
examples = [
    (1_000_000,    500,  "1M tweets × 500B each"),
    (10_000_000,   5000, "10M photos × 5KB metadata"),
    (1_000_000_000, 1000, "1B user profiles × 1KB each"),
    (500_000,      50_000_000, "500K videos × 50MB each"),
]
for items, size, desc in examples:
    print(f"  {desc:<35} = {storage_per_day(items, size)}")

## ❌ BAD: Guessing ("I think we need 50 servers")

Here's what a **bad** answer looks like in an interview:

> *"Twitter has a lot of users, so we'll need maybe... 50 application servers,
> 10 database servers, and a big cache cluster. Oh and probably some sharding."*

**Why this is bad:**
- No math, no reasoning, no justification
- Probably **wildly over-engineered** (a single Postgres handles 10K+ QPS)
- Shows no understanding of actual system capabilities
- Interviewer has no way to evaluate your thinking

In [ ]:
# ❌ BAD: The "gut feel" approach

print("❌ BAD: Capacity Planning by Guessing")
print("=" * 55)
print()
print('  Interviewer: "How would you handle 10M daily users?"')
print()
print('  Bad answer: "We probably need about 50 app servers,')
print('  10 database replicas, and Redis cluster with 20 nodes."')
print()
print("  ❌ No math")
print("  ❌ No justification")
print("  ❌ Likely 10-50× over-provisioned")
print("  ❌ Shows no understanding of hardware capabilities")
print()
print("  A single modern server has 128 vCPUs and 512 GB RAM.")
print("  That's enough for most apps you'd design in an interview!")

## ✅ BETTER: Back-of-Envelope Math

The **right approach** in an interview: start with users, derive numbers step by step.

### Framework:
1. **Users** → How many DAU?
2. **Actions** → How many actions per user per day?
3. **QPS** → daily_actions / 86,400 seconds
4. **Peak** → QPS × 3 (peak is typically 2-3× average)
5. **Servers** → peak_QPS / QPS_per_server

In [ ]:
# ✅ BETTER: Back-of-envelope calculation for a social media app

print("✅ BETTER: Back-of-Envelope Estimation")
print("=" * 55)
print()

# Step 1: Users
monthly_active_users = 100_000_000   # 100M MAU
dau_ratio = 0.5                       # 50% of MAU are daily active
dau = int(monthly_active_users * dau_ratio)

print(f"Step 1 — Users:")
print(f"  Monthly Active Users (MAU): {monthly_active_users:,}")
print(f"  DAU ratio: {dau_ratio:.0%}")
print(f"  Daily Active Users (DAU):   {dau:,}")
print()

# Step 2: Actions per user
reads_per_user_per_day = 50    # scroll feed, view profiles, etc.
writes_per_user_per_day = 2    # post, comment, like

total_reads_per_day = dau * reads_per_user_per_day
total_writes_per_day = dau * writes_per_user_per_day

print(f"Step 2 — Actions:")
print(f"  Reads per user per day:  {reads_per_user_per_day}")
print(f"  Writes per user per day: {writes_per_user_per_day}")
print(f"  Total reads/day:  {total_reads_per_day:,}")
print(f"  Total writes/day: {total_writes_per_day:,}")
print()

# Step 3: QPS
read_qps = requests_to_qps(total_reads_per_day)
write_qps = requests_to_qps(total_writes_per_day)

print(f"Step 3 — QPS (divide by 86,400):")
print(f"  Read QPS (avg):  {read_qps:,.0f}")
print(f"  Write QPS (avg): {write_qps:,.0f}")
print()

# Step 4: Peak QPS
peak_read_qps = qps_to_peak(read_qps)
peak_write_qps = qps_to_peak(write_qps)

print(f"Step 4 — Peak QPS (× 3):")
print(f"  Peak Read QPS:  {peak_read_qps:,.0f}")
print(f"  Peak Write QPS: {peak_write_qps:,.0f}")
print()

# Step 5: How many servers?
postgres_qps = 10_000    # conservative: simple queries
redis_qps = 100_000      # conservative: GET/SET operations

print(f"Step 5 — Server Requirements:")
print(f"  A single PostgreSQL handles ~{postgres_qps:,} simple QPS")
print(f"  A single Redis handles ~{redis_qps:,} GET/SET QPS")
print()
print(f"  For reads (via Redis cache):")
print(f"    {peak_read_qps:,.0f} QPS / {redis_qps:,} per Redis = {peak_read_qps/redis_qps:.1f} Redis instances")
print(f"  For writes (to PostgreSQL):")
print(f"    {peak_write_qps:,.0f} QPS / {postgres_qps:,} per PG = {peak_write_qps/postgres_qps:.1f} PostgreSQL instances")
print()
print("  💡 A 100M MAU social app can run on ~1 Redis + ~1 PostgreSQL!")
print("     This is why premature sharding is a common interview mistake.")

## 🏆 BEST: Actually Measure It (Benchmarking)

Back-of-envelope is great for interviews, but in production you **measure**.
Let's benchmark our actual PostgreSQL and Redis to see real throughput.

In [ ]:
# 🏆 BEST: Measure actual throughput of our PostgreSQL and Redis

def measure_throughput(func, label, duration_seconds=5):
    """Run a function repeatedly for N seconds and count operations."""
    count = 0
    start = time.time()
    while time.time() - start < duration_seconds:
        func()
        count += 1
    elapsed = time.time() - start
    qps = count / elapsed
    print(f"  {label}:")
    print(f"    Operations: {count:,} in {elapsed:.1f}s")
    print(f"    Throughput: {qps:,.0f} ops/sec")
    return qps

print("🏆 BEST: Measured Throughput (5-second benchmarks)")
print("=" * 55)
print()

# PostgreSQL throughput (with connection reuse)
conn = get_db_connection()
def pg_read():
    cur = conn.cursor()
    cur.execute("SELECT value FROM benchmark_kv WHERE key = 'key:5000'")
    cur.fetchone()

pg_qps = measure_throughput(pg_read, "PostgreSQL SELECT (indexed, reused conn)")
conn.close()
print()

# PostgreSQL write throughput
conn = get_db_connection()
conn.autocommit = True
counter = [0]
def pg_write():
    cur = conn.cursor()
    counter[0] += 1
    cur.execute(
        "INSERT INTO benchmark_kv (key, value) VALUES (%s, %s) ON CONFLICT (key) DO UPDATE SET value = EXCLUDED.value",
        (f"bench:{counter[0]}", f"value_{counter[0]}")
    )

pg_write_qps = measure_throughput(pg_write, "PostgreSQL INSERT/UPSERT")
conn.close()
print()

# Redis read throughput
r = get_redis_client()
r.set("bench:throughput", "test_value_123")

def redis_read():
    r.get("bench:throughput")

redis_read_qps = measure_throughput(redis_read, "Redis GET")
print()

# Redis write throughput
write_counter = [0]
def redis_write():
    write_counter[0] += 1
    r.set(f"bench:w:{write_counter[0]}", "test_value")

redis_write_qps = measure_throughput(redis_write, "Redis SET")

In [ ]:
# Compare measured vs theoretical

print()
print("📊 Measured vs Theoretical Throughput")
print("=" * 60)
print(f"{'Component':<28} {'Measured':>12} {'Theoretical':>12}")
print("-" * 60)
print(f"  {'PostgreSQL reads':<26} {pg_qps:>10,.0f}/s {'~10,000/s':>12}")
print(f"  {'PostgreSQL writes':<26} {pg_write_qps:>10,.0f}/s {'~5,000/s':>12}")
print(f"  {'Redis reads':<26} {redis_read_qps:>10,.0f}/s {'~100,000/s':>12}")
print(f"  {'Redis writes':<26} {redis_write_qps:>10,.0f}/s {'~100,000/s':>12}")
print()
print("💡 Note: We're running on Docker on localhost — production servers")
print("   with dedicated hardware will be 2-5× faster.")
print()
print("   Key insight: Redis is roughly 10× faster than PostgreSQL for")
print("   simple operations. That's why we cache hot data in Redis!")

## 💾 Storage Estimation

How much disk space does your system need? The formula is simple:

```
Daily storage = items_per_day × bytes_per_item
Yearly storage = daily_storage × 365
Total storage = yearly_storage × retention_years
```

Let's calculate for several real scenarios.

In [ ]:
# Storage estimation for different systems

print("💾 Storage Estimation Examples")
print("=" * 70)
print()

scenarios = [
    {
        "name": "Chat App (like WhatsApp)",
        "dau": 500_000_000,
        "items_per_user_per_day": 40,        # 40 messages/day
        "bytes_per_item": 500,                # avg message ~500 bytes
        "retention_years": 5,
    },
    {
        "name": "Social Media (like Twitter)",
        "dau": 200_000_000,
        "items_per_user_per_day": 0.5,        # 0.5 tweets/day avg
        "bytes_per_item": 500,                # tweet text + metadata
        "retention_years": 10,
    },
    {
        "name": "E-commerce (like Amazon)",
        "dau": 50_000_000,
        "items_per_user_per_day": 0.1,        # 1 order per 10 visits
        "bytes_per_item": 5_000,              # order with items
        "retention_years": 7,
    },
    {
        "name": "Video Platform (metadata only)",
        "dau": 500_000_000,
        "items_per_user_per_day": 0.001,      # 1 in 1000 users uploads
        "bytes_per_item": 10_000,             # video metadata
        "retention_years": 10,
    },
]

for s in scenarios:
    items_per_day = s["dau"] * s["items_per_user_per_day"]
    daily_bytes = items_per_day * s["bytes_per_item"]
    yearly_bytes = daily_bytes * 365
    total_bytes = yearly_bytes * s["retention_years"]

    print(f"📦 {s['name']}")
    print(f"   DAU: {s['dau']:,}")
    print(f"   Items/user/day: {s['items_per_user_per_day']}")
    print(f"   Bytes/item: {s['bytes_per_item']:,}")
    print(f"   ─────────────────────────────────")
    print(f"   Items/day:      {items_per_day:>15,.0f}")
    print(f"   Storage/day:    {storage_per_day(int(items_per_day), s['bytes_per_item']):>15}")
    print(f"   Storage/year:   {storage_per_day(int(items_per_day * 365), s['bytes_per_item']):>15}")
    print(f"   Total ({s['retention_years']}yr):   {storage_per_day(int(items_per_day * 365 * s['retention_years']), s['bytes_per_item']):>15}")

    # Can a single Postgres handle this?
    total_tb = total_bytes / 1e12
    if total_tb < 10:
        print(f"   ✅ Fits on a single PostgreSQL (< 10 TB)")
    elif total_tb < 100:
        print(f"   ⚠️  May need partitioning ({total_tb:.0f} TB)")
    else:
        print(f"   ❌ Needs distributed storage ({total_tb:.0f} TB)")
    print()

## 🌐 Bandwidth Estimation

Bandwidth = QPS × response_size. Let's calculate for our scenarios.

In [ ]:
# Bandwidth estimation

print("🌐 Bandwidth Estimation")
print("=" * 70)
print()

bandwidth_scenarios = [
    ("API endpoint (JSON)",       10_000,  5_000,    "5 KB JSON response"),
    ("Image thumbnails",          50_000,  50_000,   "50 KB thumbnail"),
    ("Full images",               10_000,  500_000,  "500 KB image"),
    ("Video streaming (1 viewer)", 1,      625_000,  "5 Mbps stream"),
    ("Video streaming (10K)",     10_000,  625_000,  "5 Mbps × 10K viewers"),
]

print(f"{'Scenario':<30} {'QPS':>8} {'Size':>10} {'Bandwidth':>12} {'vs 25Gbps':>10}")
print("-" * 78)

for name, qps, size_bytes, desc in bandwidth_scenarios:
    bw_mbps = bandwidth(qps, size_bytes)
    bw_gbps = bw_mbps / 1000
    pct_of_25gbps = (bw_gbps / 25) * 100
    print(f"  {name:<28} {qps:>8,} {desc:>18} {bw_mbps:>8,.0f} Mbps  {pct_of_25gbps:>6.1f}%")

print()
print("💡 A standard server has 25 Gbps network capacity.")
print("   Most API workloads use < 1% of available bandwidth.")
print("   Video streaming is the exception — it's bandwidth-hungry!")
print("   That's why YouTube uses CDNs (Content Delivery Networks).")

## 🧹 Cleanup

In [ ]:
# Clean up benchmark keys
r = get_redis_client()
keys = r.keys("bench:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned up {len(keys)} Redis keys")

# Clean up benchmark rows in Postgres
conn = get_db_connection()
conn.autocommit = True
cur = conn.cursor()
cur.execute("DELETE FROM benchmark_kv WHERE key LIKE 'bench:%'")
print(f"🧹 Cleaned up {cur.rowcount} benchmark rows from PostgreSQL")
conn.close()

## 📚 Summary

### The Three Approaches

| Approach | When to Use | Quality |
|----------|------------|---------|
| ❌ **Guessing** | Never | Shows no understanding |
| ✅ **Back-of-envelope** | Interviews, early design | Shows structured thinking |
| 🏆 **Benchmarking** | Production planning | Shows engineering rigor |

### Key Numbers to Memorize

| Fact | Value |
|------|-------|
| Seconds in a day | ~86K ≈ 100K |
| 1M requests/day | ~12 QPS |
| Single PostgreSQL (simple reads) | ~10K QPS |
| Single Redis | ~100K QPS |
| Modern server RAM | up to 512 GB (common) to 24 TB (max) |
| Modern server SSD | up to 60 TB |
| Network per server | 25 Gbps (standard), 100 Gbps (high perf) |

### Common Mistake: Premature Sharding

A single PostgreSQL handles:
- **10 TB+** of data
- **10K+ simple queries/sec**
- **5K+ writes/sec**

Most interview-scale systems fit on **one database**. Don't shard unless the math demands it!

### Next Up

In **Notebook 3**, we'll put everything together and practice **full back-of-envelope estimations**
for Twitter, YouTube, and Uber — step by step, the way you'd do it in an interview.